# MSME Financial Health - Heavy Production Pipeline (Dual T4 GPU)
**Architecture:** Multi-Layer Stacking Ensemble via AutoGluon, integrating CatBoost, XGBoost, and PyTorch DL architectures.
**Objective:** Maximize ROC-AUC on highly variant MSME financial markers.

In [ ]:
!pip install -q -U pip setuptools wheel


In [ ]:
!pip install -q scikit-learn autogluon catboost lightgbm torch optuna shap 


In [ ]:

import os
import gc
import pandas as pd
import numpy as np
import shap
import warnings
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
shap.initjs()

# Verify GPU Availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")

### 1. Data Ingestion & Memory Optimization
Downcasting float64 to float32 to prevent OOM errors during extreme model stacking.

In [ ]:
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            else:
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    print(f"Memory optimized from {start_mem:.2f} MB to {df.memory_usage().sum() / 1024**2:.2f} MB")
    return df

# Load and optimize dataset
# data_path = '/kaggle/input/datasets/chauhanrj/small-msme-dataset/msme_final_engineered_features.csv'
# df = pd.read_csv(data_path)
# df = reduce_mem_usage(df)

# # IMPORTANT: Update this to your generated target column name
# target_column = 'is_defaulter' 

# # 80/10/10 Split for Train/Val/Test
# train_data, temp_data = train_test_split(df, test_size=0.20, random_state=42, stratify=df[target_column])
# val_data, test_data = train_test_split(temp_data, test_size=0.50, random_state=42, stratify=temp_data[target_column])

# train_data = TabularDataset(train_data)
# val_data = TabularDataset(val_data)
# test_data = TabularDataset(test_data)

# print(f"Training set: {train_data.shape} | Validation set: {val_data.shape} | Test set: {test_data.shape}")

### 2. AutoGluon Multi-GPU Heavy Architecture
Configuring custom GPU algorithms and forcing multi-layer stacking.

### 3. Comprehensive Evaluation

In [ ]:
# from autogluon.tabular import TabularPredictor

# # Point to the dataset path in /kaggle/input/
# v1_model_path = '/kaggle/input/datasets/chauhanrj/idbi-msme-autogluon-v1/agModels-MSME-DualT4'

# # Load it instantly!
# predictor = TabularPredictor.load(v1_model_path)

## Automated Data Analysis (EDA)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from autogluon.tabular import TabularDataset

# 1. Load Data
data_path = '/kaggle/input/datasets/chauhanrj/small-msme-dataset/msme_final_engineered_features.csv'
df = pd.read_csv(data_path)
target_column = 'is_defaulter'

# 2. Drop all known leaky and ID features to force the model to learn real finance patterns
columns_to_drop = ['risk_band', 'msme_id', 'company_name', 'bounce_mandate_failure_rate']
df_clean = df.drop(columns=columns_to_drop, errors='ignore')

# Save splits for training
train_df, test_df = train_test_split(df_clean, test_size=0.10, random_state=42, stratify=df_clean[target_column])
train_data = TabularDataset(train_df)
test_data = TabularDataset(test_df)

print("--- RUNNING DATASET HEALTH & LEAKAGE CHECK ---")

# Convert categorical columns to numbers temporarily for a pure mathematical check
df_numeric = df_clean.copy()
for col in df_numeric.select_dtypes(include=['object']).columns:
    df_numeric[col] = LabelEncoder().fit_transform(df_numeric[col].astype(str))

# 3. Correlation Check
correlations = df_numeric.corr()[target_column].abs().sort_values(ascending=False)
print("\n🔍 Top Feature Correlations with Target:")
print("⚠️ Watch out for any feature with a score > 0.80")
print(correlations.head(10).to_string())

# 4. Quick Feature Importance Check
X = df_numeric.drop(columns=[target_column]).fillna(0)
y = df_numeric[target_column]

rf = RandomForestClassifier(n_estimators=50, max_depth=5, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\n📊 Quick Feature Importance Profile:")
print("⚠️ Watch out for any single feature dominating above 0.50 (50%)")
print(importances.head(10).to_string())

# 5. Visualize the Distribution of Feature Influence
plt.figure(figsize=(10, 5))
sns.barplot(x=importances.head(10).values, y=importances.head(10).index, palette='viridis')
plt.title("Feature Influence Profile (Target Leakage Audit)")
plt.xlabel("Importance Score")
plt.tight_layout()
plt.show()

In [ ]:
import gc
import os
import shutil
from autogluon.tabular import TabularPredictor

# Reset the previous model directory state
save_path = 'agModels-MSME-DualT4'
if os.path.exists(save_path):
    shutil.rmtree(save_path)

total_rows = len(train_data)
if total_rows < 1000:
    stack_levels = 0
    bag_folds = 0
    presets = 'medium_quality'
else:
    stack_levels = 2
    bag_folds = 5
    presets = 'best_quality'

# 3. Initialize and fit the heavy stacker with AutoGluon's native feature processing
predictor = TabularPredictor(label=target_column, eval_metric='roc_auc', path=save_path).fit(
    train_data=train_data,
    tuning_data=None,               
    presets=presets,
    dynamic_stacking=False,         
    num_stack_levels=stack_levels,             
    num_bag_folds=bag_folds,                
    ag_args_fit={'num_gpus': 2},    
    ag_args_ensemble={'fold_fitting_strategy': 'sequential_local'}, 
    excluded_model_types=['LGB'],   # Bypasses the broken Dask/PyArrow Kaggle environment bug
    time_limit=14400,               
    verbosity=2                     
)

gc.collect()
print("\nModel trained perfectly with balanced feature engineering rules!")

In [ ]:
# import gc
# import os
# import shutil
# import pandas as pd
# from autogluon.tabular import TabularDataset, TabularPredictor
# from sklearn.model_selection import train_test_split

# # 1. Clear previous corrupted runs
# save_path = 'agModels-MSME-DualT4'
# if os.path.exists(save_path):
#     print(f"Removing legacy directory: {save_path} to reset learner state...")
#     shutil.rmtree(save_path)

# # 2. Data Loading & Preparation
# data_path = '/kaggle/input/datasets/chauhanrj/small-msme-dataset/msme_final_engineered_features.csv'
# df = pd.read_csv(data_path)

# if 'reduce_mem_usage' in globals():
#     df = reduce_mem_usage(df)

# target_column = 'is_defaulter' 

# # --- THE CRITICAL FIX: DROPPING LEAKAGE AND ID COLUMNS ---
# columns_to_drop = ['risk_band', 'msme_id', 'company_name']
# print(f"Dropping columns to prevent target leakage and overfitting: {columns_to_drop}")
# df = df.drop(columns=columns_to_drop, errors='ignore')

# # Keep a clean 10% completely unseen for final hold-out evaluation
# train_heavy_df, test_df = train_test_split(df, test_size=0.10, random_state=42, stratify=df[target_column])

# train_heavy_data = TabularDataset(train_heavy_df)
# test_data = TabularDataset(test_df)

# print(f"Unified Training Pool: {train_heavy_data.shape}")
# print(f"Isolated Test Set: {test_data.shape}")

# # --- ADAPTIVE CONFIGURATION LOGIC ---
# total_rows = len(train_heavy_data)
# if total_rows < 1000:
#     print("⚠️ WARNING: Tiny dataset detected. Disabling deep stacking to prevent crashes.")
#     stack_levels = 0
#     bag_folds = 0
#     presets = 'medium_quality'
# else:
#     print("✅ Large dataset detected. Enabling extreme heavy stacking for max accuracy.")
#     stack_levels = 2
#     bag_folds = 5
#     presets = 'best_quality'

# # 3. Initialize and fit the heavy stacker
# predictor = TabularPredictor(label=target_column, eval_metric='roc_auc', path=save_path).fit(
#     train_data=train_heavy_data,
#     tuning_data=None,               
#     presets=presets,
#     dynamic_stacking=False,         
#     num_stack_levels=stack_levels,             
#     num_bag_folds=bag_folds,                
#     ag_args_fit={'num_gpus': 2},    
#     ag_args_ensemble={'fold_fitting_strategy': 'sequential_local'}, 
#     excluded_model_types=['LGB'],   
#     time_limit=14400,               
#     verbosity=2                     
# )

# gc.collect()
# print("\nTraining completed successfully without target leakage!")

In [ ]:
# Test Hold-out unseen data
results = predictor.evaluate(test_data, silent=True)
print("\n--- Unseen Data Performance ---")
for metric, score in results.items():
    print(f"{metric}: {score:.4f}")

# Display the ensemble architecture leaderboard
leaderboard = predictor.leaderboard(test_data, silent=True)
display(leaderboard.head(15))

# Feature Importance Extraction
fi = predictor.feature_importance(test_data)
display(fi.head(10))

### 4. Global Interperatability (SHAP)
Visualizing feature impact across the highest-weighted models.

In [ ]:
import shap
import pandas as pd
import numpy as np

class AutogluonShapWrapper:
    def __init__(self, predictor, reference_df):
        self.ag_model = predictor
        self.feature_names = reference_df.columns
        # Remember the exact original data types
        self.dtypes = reference_df.dtypes.to_dict() 
        
    def predict(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X, columns=self.feature_names)
            
            # SHAP strips Pandas data types and turns everything into generic objects.
            # AutoGluon hates this. We must restore the original types here:
            for col, dtype in self.dtypes.items():
                try:
                    X[col] = X[col].astype(dtype)
                except Exception:
                    pass # Fallback for any synthetic categorical anomalies 
                    
        return self.ag_model.predict_proba(X).iloc[:, 1].values


X_test_explain = test_data.drop(columns=[target_column])

# Initialize the upgraded wrapper
ag_wrapper = AutogluonShapWrapper(predictor, X_test_explain)

# FIX: Use shap.sample instead of shap.kmeans to bypass the text math crash
background = shap.sample(X_test_explain, 50)
explainer = shap.KernelExplainer(ag_wrapper.predict, background)

# Calculate SHAP for a sample of testing data
# WARNING: KernelExplainer is slow on AutoGluon ensembles. 
# Starting with 50 rows to test. If it runs fast, you can increase it back to 150.
sample_size = 50 
shap_values = explainer.shap_values(X_test_explain.iloc[:sample_size])

shap.summary_plot(shap_values, X_test_explain.iloc[:sample_size])